# 📊 Olist Lakehouse — Exploratory Data Analysis (EDA) Profiling
> **Tác giả:** Anh Tú (IT Lecturer & Technical Writer)  
> **Phương pháp:** Pedagogical Approach — Chuyển hóa kiến thức phức tạp thành trực quan, dễ hiểu.  
> **Môi trường:** Apache Spark 3.5.0 + PostgreSQL 16 + MinIO S3 (chạy trên container `olist_spark_notebook:8888`)  
> **Bản quyền:** *Made by Anh Tu - Share to be share*  

---

## 💡 TẠI SAO DATA ENGINEER CẦN EDA CHUYÊN BIỆT?
Khác với **Data Science EDA** (tìm kiếm tương quan `correlation`, phân phối chuẩn để huấn luyện mô hình ML), **Data Engineering EDA** là quá trình khám nghiệm cấu trúc và ranh giới dữ liệu (**Structural & Boundary Inspection**) nhằm:
1. **Thiết lập Data Contract:** Định hình kích thước kiểu dữ liệu chính xác (VARCHAR sizing, tránh tràn bộ nhớ khi Spark shuffle).
2. **Xây dựng Data Quality Rule Catalog:** Không đặt bừa rule `NOT NULL` hay `> 0` làm sập pipeline (Pipeline Crash), mà xác lập đúng chính sách (*Quarantine, Log Warning, Unknown Value*).
3. **Phát hiện dị thường xuyên nguồn (Cross-Source Isolation):** Đo lường tính toàn vẹn tham chiếu giữa RDBMS OLTP và Object Storage Landing.

## 1. Khởi Tạo SparkSession Phân Tán Kết Nối PostgreSQL JDBC & MinIO S3A
Spark đóng vai trò là Distributed Compute Engine kết nối đồng thời 2 nguồn:
- **Nguồn 1:** PostgreSQL OLTP qua JDBC Driver (`org.postgresql:postgresql:42.7.3`).
- **Nguồn 2:** MinIO S3 Landing qua S3A Filesystem (`org.apache.hadoop:hadoop-aws:3.3.4`).

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

packages = [
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    "org.postgresql:postgresql:42.7.3"
]

spark = SparkSession.builder \
    .appName("Olist-EDA-Profiling-5Pillars") \
    .master("local[2]") \
    .config("spark.driver.memory", "1g") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("MINIO_ENDPOINT", "http://minio:9000")) \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✔ SparkSession đã khởi tạo thành công với S3A Connector và PostgreSQL JDBC Driver!")

✔ SparkSession đã khởi tạo thành công với S3A Connector và PostgreSQL JDBC Driver!


## 2. Nguồn 1: Trích Xuất 5 Bảng Transactional Từ PostgreSQL OLTP
Kết nối Supabase Session Pooler qua Spark JDBC để trích xuất mẻ dữ liệu B0:
`orders`, `order_items`, `order_payments`, `customers`, `customer_pii`.

In [2]:
pg_host = os.getenv("POSTGRES_HOST", "aws-1-ap-southeast-2.pooler.supabase.com")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_user = os.getenv("POSTGRES_USER", "postgres.iuvjhphnqpekptjzsifm")
pg_pwd = os.getenv("POSTGRES_PASSWORD", "olist-lakehouse")
pg_db = os.getenv("POSTGRES_DB", "postgres")

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

def load_pg_table(tbl_name):
    return spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", tbl_name) \
        .option("user", pg_user) \
        .option("password", pg_pwd) \
        .option("driver", "org.postgresql.Driver") \
        .load()

df_orders = load_pg_table("orders")
df_items = load_pg_table("order_items")
df_payments = load_pg_table("order_payments")
df_customers = load_pg_table("customers")
df_pii = load_pg_table("customer_pii")

print(f"✔ Đọc thành công 5 bảng PostgreSQL OLTP:")
print(f"  • orders         : {df_orders.count():>8,} dòng")
print(f"  • order_items    : {df_items.count():>8,} dòng")
print(f"  • order_payments : {df_payments.count():>8,} dòng")
print(f"  • customers      : {df_customers.count():>8,} dòng")
print(f"  • customer_pii   : {df_pii.count():>8,} dòng")

✔ Đọc thành công 5 bảng PostgreSQL OLTP:
  • orders         :   14,940 dòng
  • order_items    :   16,679 dòng
  • order_payments :   15,870 dòng
  • customers      :   99,441 dòng
  • customer_pii   :   96,096 dòng


## 3. Nguồn 2: Trích Xuất 5 Tệp Tham Chiếu Từ MinIO Landing S3
Đọc trực tiếp các tệp CSV từ bucket `s3a://landing/` qua giao thức S3A phân tán:
`products`, `sellers`, `order_reviews`, `geolocation`, `translation`.

In [3]:
def load_s3_csv(key):
    return spark.read.option("header", "true").csv(f"s3a://landing/{key}")

df_products = load_s3_csv("ref/olist_products_dataset.csv")
df_sellers = load_s3_csv("ref/olist_sellers_dataset.csv")
df_reviews = load_s3_csv("reviews/olist_order_reviews_dataset.csv")
df_geo = load_s3_csv("ref/olist_geolocation_dataset.csv")
df_trans = load_s3_csv("ref/product_category_name_translation.csv")

print(f"✔ Đọc thành công 5 tệp CSV từ MinIO Landing S3:")
print(f"  • products     : {df_products.count():>10,} dòng")
print(f"  • sellers      : {df_sellers.count():>10,} dòng")
print(f"  • reviews      : {df_reviews.count():>10,} dòng")
print(f"  • geolocation  : {df_geo.count():>10,} dòng")
print(f"  • translations : {df_trans.count():>10,} dòng")

✔ Đọc thành công 5 tệp CSV từ MinIO Landing S3:
  • products     :     32,951 dòng
  • sellers      :      3,095 dòng
  • reviews      :    104,162 dòng
  • geolocation  :  1,000,163 dòng
  • translations :         71 dòng


## 4. Trụ Cột 1: Schema Sizing & Storage Footprint
Đo lường độ dài chuỗi (`min_len`, `max_len`, `avg_len`) để định hình chính xác kích thước VARCHAR, tránh dùng generic `TEXT` hoặc `VARCHAR(255)` gây lãng phí bộ nhớ khi Spark thực hiện shuffle join.

In [4]:
df_sizing = df_customers.select(
    F.min(F.length("customer_id")).alias("min_cust_id"),
    F.max(F.length("customer_id")).alias("max_cust_id"),
    F.min(F.length("customer_state")).alias("min_state"),
    F.max(F.length("customer_state")).alias("max_state"),
    F.round(F.avg(F.length("customer_city")), 1).alias("avg_city_len"),
    F.max(F.length("customer_city")).alias("max_city_len")
)
df_sizing.show()
print("💡 KHUYẾN NGHỊ DATA CONTRACT:")
print("  - customer_id    : VARCHAR(32) (Độ dài cố định chính xác 32 ký tự)")
print("  - customer_state : VARCHAR(2)  (Mã bang Brazil chuẩn 2 ký tự)")
print("  - customer_city  : VARCHAR(32) (Thành phố dài nhất 32 ký tự)")

+-----------+-----------+---------+---------+------------+------------+
|min_cust_id|max_cust_id|min_state|max_state|avg_city_len|max_city_len|
+-----------+-----------+---------+---------+------------+------------+
|         32|         32|        2|        2|        10.3|          32|
+-----------+-----------+---------+---------+------------+------------+

💡 KHUYẾN NGHỊ DATA CONTRACT:
  - customer_id    : VARCHAR(32) (Độ dài cố định chính xác 32 ký tự)
  - customer_state : VARCHAR(2)  (Mã bang Brazil chuẩn 2 ký tự)
  - customer_city  : VARCHAR(32) (Thành phố dài nhất 32 ký tự)


## 5. Trụ Cột 2: Domain Boundaries & Dị Thường Số Học
Khảo sát các trường số để phát hiện giá trị âm và các trường hợp thanh toán bằng 0 BRL.

In [5]:
df_anomalies = df_payments.select(
    F.min("payment_value").alias("min_payment"),
    F.max("payment_value").alias("max_payment"),
    F.sum(F.when(F.col("payment_value") == 0, 1).otherwise(0)).alias("zero_payment_count"),
    F.sum(F.when(F.col("payment_value") < 0, 1).otherwise(0)).alias("negative_payment_count")
)
df_anomalies.show()

# Truy vết cụ thể 9 bản ghi payment_value = 0
print("🔍 Chi tiết các giao dịch thanh toán 0 BRL (Khách hàng áp voucher giảm giá 100%):")
df_payments.filter(F.col("payment_value") == 0).show()

+-----------+-----------+------------------+----------------------+
|min_payment|max_payment|zero_payment_count|negative_payment_count|
+-----------+-----------+------------------+----------------------+
|        0.0|    6929.31|                 4|                     0|
+-----------+-----------+------------------+----------------------+

🔍 Chi tiết các giao dịch thanh toán 0 BRL (Khách hàng áp voucher giảm giá 100%):
+--------------------+------------------+------------+--------------------+-------------+-------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|         updated_at|
+--------------------+------------------+------------+--------------------+-------------+-------------------+
|fa65dad1b0e818e3c...|                14|     voucher|                   1|          0.0|2017-04-20 12:45:34|
|45ed6e85398a87c25...|                 3|     voucher|                   1|          0.0|2017-06-08 21:18:45|
|fa65dad1b0e818e3c...|      

## 6. Trụ Cột 3: Temporal Consistency & Kiểm Tra Giao Trễ
Đo lường dải thời gian mua sắm và tỷ lệ đơn hàng bị giao trễ so với dự kiến (`order_delivered_customer_date > order_estimated_delivery_date`).

In [6]:
df_delivered = df_orders.filter(
    (F.col("order_status") == "delivered") &
    F.col("order_delivered_customer_date").isNotNull() &
    F.col("order_estimated_delivery_date").isNotNull()
)

total_delivered = df_delivered.count()
late_delivered = df_delivered.filter(
    F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date")
).count()

late_rate = round(late_delivered * 100.0 / total_delivered, 2)
print(f"Tổng số đơn giao thành công : {total_delivered:,}")
print(f"Số đơn bị giao trễ hẹn     : {late_delivered:,}")
print(f"Tỷ lệ giao trễ (SLA breach): {late_rate}%")

Tổng số đơn giao thành công : 14,199
Số đơn bị giao trễ hẹn     : 652
Tỷ lệ giao trễ (SLA breach): 4.59%


## 7. Trụ Cột 4: Cardinality & Hiện Tượng Thổi Phồng Khách Hàng
So sánh số lượng `customer_id` (mã giao dịch đơn) và `customer_unique_id` (cá nhân thực tế).

In [7]:
total_cust_id = df_customers.select("customer_id").distinct().count()
total_unique_id = df_customers.select("customer_unique_id").distinct().count()
inflation_diff = total_cust_id - total_unique_id
inflation_pct = round(inflation_diff * 100.0 / total_cust_id, 2)

print(f"Số lượng transaction customer_id : {total_cust_id:,}")
print(f"Số lượng người dùng thực tế     : {total_unique_id:,}")
print(f"Chênh lệch mã khách lặp lại     : {inflation_diff:,} ({inflation_pct}%)")
print("🚨 LƯU Ý SỐNG CÒN KHI TÍNH LTV / RFM:")
print("  Bắt buộc phải GROUP BY theo customer_unique_id! Nếu dùng customer_id,")
print("  tần suất mua hàng (Frequency) của mọi khách hàng sẽ luôn bằng 1!")

Số lượng transaction customer_id : 99,441
Số lượng người dùng thực tế     : 96,096
Chênh lệch mã khách lặp lại     : 3,345 (3.36%)
🚨 LƯU Ý SỐNG CÒN KHI TÍNH LTV / RFM:
  Bắt buộc phải GROUP BY theo customer_unique_id! Nếu dùng customer_id,
  tần suất mua hàng (Frequency) của mọi khách hàng sẽ luôn bằng 1!


## 8. Trụ Cột 5: Cross-Source Referential Integrity (Spark Join)
Kiểm tra xem các mặt hàng trong `order_items` (PostgreSQL) có chứa `product_id` nào không tồn tại trong danh mục `products` (MinIO S3) hay không.

In [8]:
# Thực hiện Left Anti Join giữa PostgreSQL order_items và MinIO Landing products
df_orphan_products = df_items.join(
    df_products,
    df_items["product_id"] == df_products["product_id"],
    "left_anti"
)

orphan_count = df_orphan_products.count()
print(f"Số dòng order_items mồ côi (không tìm thấy trong catalog S3): {orphan_count}")
if orphan_count > 0:
    print("Mẫu các mặt hàng mồ côi:")
    df_orphan_products.select("order_id", "order_item_id", "product_id", "price").show(5)
    print("💡 ĐỀ XUẤT PIPELINE:")
    print("  Ở tầng Gold mô hình chiều, ánh xạ các sản phẩm này về dim_product với product_sk = -1 (Unknown Product)!")

Số dòng order_items mồ côi (không tìm thấy trong catalog S3): 0


## 9. Tổng Kết & Khuyến Nghị Data Contract Cho Tầng Bronze/Silver
1. **Schema Sizing:** Cố định `VARCHAR(32)` cho các UUID/MD5 hash, `VARCHAR(2)` cho bang.
2. **Data Quality Rule:** Cho phép `payment_value = 0` (ngưỡng ≤ 0.01% mẻ nạp), cách ly các bản ghi âm tiền.
3. **Mô Hình Chiều Gold:** Ánh xạ khách hàng theo `customer_unique_id` (SCD Type 2) và gán `product_sk = -1` cho các sản phẩm khuyết thiếu.

---
*Made by Anh Tu - Share to be share*